# Tối ưu hóa đường đi trong mê cung bằng thuật toán A* và phân tích đa mục tiêu
Đồ án môn Nhập môn tối ưu hóa. Chạy các cell theo thứ tự. File này minh họa thuật toán; `run_project.py` chạy toàn bộ thực nghiệm.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.maze import generate_maze
from src.algorithms import solve_bfs, solve_dijkstra, solve_astar, validate_path
import matplotlib.pyplot as plt

## 1. Sinh mê cung
Ô `1` là tường, ô `0` là lối đi. Điểm đầu là (1,1), đích là (size-2,size-2).

In [ ]:
maze = generate_maze(size=21, seed=2026, loop_rate=0.14)
plt.figure(figsize=(5,5))
plt.imshow(maze["grid"], cmap="Greys")
plt.title("Mê cung 21×21")
plt.axis("off")
plt.show()

## 2. Tìm đường bằng BFS và A*
BFS đảm bảo ngắn nhất với chi phí mỗi bước = 1. A* dùng khoảng cách Manhattan; w=1 vẫn tìm ngắn nhất.

In [ ]:
bfs = solve_bfs(maze)
a_star = solve_astar(maze, weight=1.0)
weighted = solve_astar(maze, weight=1.3)
print("Số bước BFS / A* / Weighted A*:", bfs["steps"], a_star["steps"], weighted["steps"])
print("Số ô mở rộng:", bfs["expanded"], a_star["expanded"], weighted["expanded"])
assert bfs["steps"] == a_star["steps"] == solve_dijkstra(maze)["steps"]
assert all(validate_path(maze, s["path"]) for s in (bfs,a_star,weighted))

In [ ]:
plt.figure(figsize=(5,5))
plt.imshow(maze["grid"], cmap="Greys")
for s, color, label in [(bfs,"#ee9b4b","BFS"),(weighted,"#168d87","Weighted A*")]:
    ys = [p[0] for p in s["path"]]
    xs = [p[1] for p in s["path"]]
    plt.plot(xs,ys,color=color,linewidth=2,label=label)
plt.scatter([1,19],[1,19], c=["green","red"], s=80)
plt.legend()
plt.axis("off")
plt.show()

## 3. Xem kết quả thực nghiệm đã chạy
`run_project.py` lưu thử nghiệm Grid, Random, Pareto và test vào `results/`. **Không** lựa chọn siêu tham số dựa vào test.

In [ ]:
summary = json.loads((ROOT / "results" / "summary.json").read_text(encoding="utf-8"))
print("Cấu hình được chọn trên development:", summary["selected_from_development"])
print("Kết quả test:")
for method, result in summary["test"].items():
    print(method, "steps=", result["mean_steps"], "expanded=", result["mean_expanded"], "optimal=", result["optimal_rate"])

In [ ]:
from IPython.display import display, Image
for name in ("02_search_history.png", "03_pareto.png"):
    display(Image(filename=str(ROOT / "figures" / name), width=700))

## 4. Kết luận
Khi bắt buộc tìm đường **ngắn nhất**, chọn BFS hoặc A* (w=1). Weighted A* chỉ phù hợp nếu được phép đánh đổi một ít độ dài để giảm công sức tìm kiếm.